In [0]:
%sql
-- =============================================================================
-- Q1: mean/stddev of annual US population, 2013-2018 (SQL primary)
-- =============================================================================
SELECT
  ROUND(AVG(population), 2) AS mean_population,
  ROUND(STDDEV(population), 2) AS stddev_population
FROM rearc_bls.silver.silver_population
WHERE year BETWEEN 2013 AND 2018;


In [0]:
%sql

-- =============================================================================
-- Q2: best year per series_id, with human-readable label (SQL alternative --
-- gold_best_year_per_series in gold.py, using a PySpark Window function, is
-- primary for this question)
-- =============================================================================
WITH yearly_sums AS (
  SELECT series_id, year, SUM(value) AS total_value
  FROM rearc_bls.silver.silver_bls_pr
  GROUP BY series_id, year
),
ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY series_id ORDER BY total_value DESC, year ASC) AS rn
  FROM yearly_sums
),
decoded AS (
  SELECT
    s.series_id,
    CONCAT_WS(' - ', sec.sector_name, c.class_text, m.measure_text, d.duration_text) AS human_label
  FROM rearc_bls.silver.silver_bls_series s
  LEFT JOIN rearc_bls.silver.silver_bls_sector sec ON s.sector_code = sec.sector_code
  LEFT JOIN rearc_bls.silver.silver_bls_class c ON s.class_code = c.class_code
  LEFT JOIN rearc_bls.silver.silver_bls_measure m ON s.measure_code = m.measure_code
  LEFT JOIN rearc_bls.silver.silver_bls_duration d ON s.duration_code = d.duration_code
)
SELECT
  r.series_id,
  dc.human_label,
  r.year AS best_year,
  r.total_value
FROM ranked r
LEFT JOIN decoded dc ON r.series_id = dc.series_id
WHERE r.rn = 1;



In [0]:
%sql
-- =============================================================================
-- Q3: PRS30006032, period Q01, value by year joined with population (SQL primary)
-- =============================================================================
SELECT
  b.year,
  b.value,
  p.population
FROM rearc_bls.silver.silver_bls_pr b
LEFT JOIN rearc_bls.silver.silver_population p ON b.year = p.year
WHERE b.series_id = 'PRS30006032' AND b.period = 'Q01'
ORDER BY b.year;